In [1]:
import tangram as tg
import numpy as np
import scanpy as sc
import pandas as pd
import torch
import os
os.environ["NUMEXPR_MAX_THREADS"] = "24" # Setting CPU threads
print("CUDA Available:", torch.cuda.is_available())
print("Current Device:", torch.cuda.current_device())
print("Device Name:", torch.cuda.get_device_name(torch.cuda.current_device()))

/home/momo/miniforge3/envs/tangram-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA Available: True
Current Device: 0
Device Name: NVIDIA A100 80GB PCIe


In [2]:
# Loading the spatial dataset
ad_sp = sc.read_10x_mtx(
    "/data/MERFISH/20241108_Bartelle_MouseMERFISH_LC/qi2labdatastore/mtx_output",  # the directory with the `.mtx` file
    var_names="gene_symbols",  # use gene symbols for the variable names (variables-axis index)
)

In [3]:
# Loading the reference scRNA dataset
ad_sc = sc.read_h5ad(
    '/data/scRNA/RNA/scRNA/Hammond/preprocessing/GSE121654_T_P100_LC_common_concat_zero_filter_MTremoved.h5ad'
)

In [4]:
'''
Preprocessing the spatial data
'''
# Filter out cells with too few genes
sc.pp.filter_cells(ad_sp, min_genes=5)
sc.pp.filter_genes(ad_sp, min_cells=5)
sc.pp.normalize_total(ad_sp, target_sum=1e4)


In [5]:
tg.pp_adatas(ad_sc, ad_sp, genes=None)

INFO:root:111 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:111 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.


In [6]:
ad_map = tg.map_cells_to_space(
               ad_sc,
               ad_sp,
               mode='cells',
               device = "cuda")

INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 111 genes and rna_count_based density_prior in cells mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.242, KL reg: 0.000
Score: 0.736, KL reg: 0.011
Score: 0.751, KL reg: 0.010
Score: 0.754, KL reg: 0.010
Score: 0.755, KL reg: 0.010
Score: 0.756, KL reg: 0.010
Score: 0.756, KL reg: 0.010
Score: 0.756, KL reg: 0.010
Score: 0.757, KL reg: 0.009
Score: 0.757, KL reg: 0.009


INFO:root:Saving results..


In [7]:
ad_map.write_h5ad(
    "/data/MERFISH/20241108_Bartelle_MouseMERFISH_LC/qi2labdatastore/mtx_output/Imputed_map.h5ad"
)

In [ ]:
ad_map = sc.read_h5ad(
    "/data/MERFISH/20241108_Bartelle_MouseMERFISH_LC/qi2labdatastore/mtx_output/Imputed_map.h5ad"
)

In [8]:
ad_map

AnnData object with n_obs × n_vars = 27177 × 7226
    obs: 'batch', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'n_genes', 'uniform_density', 'rna_count_based_density'
    uns: 'train_genes_df', 'training_history'

In [9]:
ad_ge = tg.project_genes(
                ad_map,
                ad_sc)

In [10]:
ad_ge

AnnData object with n_obs × n_vars = 7226 × 10570
    obs: 'n_genes', 'uniform_density', 'rna_count_based_density'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'sparsity', 'is_training'
    uns: 'training_genes', 'overlap_genes'

In [11]:
ad_ge.write_h5ad(
    "/data/MERFISH/20241108_Bartelle_MouseMERFISH_LC/qi2labdatastore/mtx_output/Imputed_Hammond.h5ad"
)